# Random Forest Training — Neuromorphic Gestures

**LEAP Project** — Classification of 6 hand gestures from spikes generated by Izhikevich models.

---

### Pipeline
1. Load spike data (features: counts on 5 fingers `N1`–`N5`)
2. **Grid Search** with 5-Fold stratified Cross-Validation for each neuron
3. Train final model with optimal parameters
4. Save in `.pkl` format (for LabVIEW Python Node)
5. Performance comparison **with Rooting** vs **without Rooting**
6. Export a summary Excel report

### Classes
| ID | Gesture |
|:---:|---|
| 1 | Open palm |
| 2 | Fist |
| 3 | Thumb only |
| 4 | Pinky only |
| 5 | Thumb + pinky |
| 6 | Horns |

---
## 0. Imports and Configuration

In [ ]:
import os, glob, re
import pandas as pd
import numpy as np
import joblib
from io import StringIO
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

In [ ]:
SCRIPT_DIR    = os.getcwd()
ROOTING_DIR   = os.path.join(SCRIPT_DIR, "Dataset", "blind_channel")
NOROOTING_DIR = os.path.join(SCRIPT_DIR, "Dataset", "route_node")
OUTPUT_DIR    = os.path.join(SCRIPT_DIR, "..", "Models", "Neuromorphic")

os.makedirs(os.path.join(OUTPUT_DIR, "model_blind"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "model_route_node"), exist_ok=True)

NEURON_COLS = ['N1', 'N2', 'N3', 'N4', 'N5']

print(f'Dataset WITH rooting:    {ROOTING_DIR}')
print(f'Dataset WITHOUT rooting: {NOROOTING_DIR}')
print(f'Model output:            {OUTPUT_DIR}')

---
## 1. Hyperparameter Grid

| Hyperparameter | Values | Description |
|---|---|---|
| `n_estimators` | 50, 100, 200 | Number of trees |
| `max_depth` | None, 10, 20 | Maximum depth |
| `min_samples_split` | 2, 5 | Minimum samples for split |
| `min_samples_leaf` | 1, 2 | Minimum samples for leaf |

Fixed parameters: `class_weight='balanced'`, `random_state=42`

**Total:** 3 × 3 × 2 × 2 = **36 combinations** per neuron, each evaluated with 5-Fold CV.

In [ ]:
PARAM_GRID = {
    'n_estimators':     [50, 100, 200],
    'max_depth':        [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2]
}

n_comb = 1
for v in PARAM_GRID.values():
    n_comb *= len(v)
print(f'Total combinations per neuron: {n_comb}')
print(f'Trainings per neuron (x5 fold): {n_comb * 5}')

---
## 2. Data Loading Functions

In [ ]:
def fix_decimal_commas(filepath):
    """Fixes CSV file with decimal comma (15 raw columns -> 8 real columns)."""
    with open(filepath, 'r') as f:
        lines = f.readlines()
    n_cols = len(lines[0].strip().split(','))
    if n_cols == 15:
        new_lines = []
        for line in lines:
            p = line.strip().split(',')
            row = (f'{p[0]}.{p[1]},{p[2]},'
                   f'{p[3]}.{p[4]},{p[5]}.{p[6]},{p[7]}.{p[8]},'
                   f'{p[9]}.{p[10]},{p[11]}.{p[12]},{p[13]}.{p[14]}')
            new_lines.append(row)
        return new_lines
    return None

In [ ]:
def extract_neuron_name(filename):
    """Extracts the neuron type from the filename."""
    basename = os.path.basename(filename).replace('.csv', '')
    for prefix in ['SpikeBD_', 'SpikeB_', 'SpikeN_', 'SpikeT_', 'Spike_']:
        if basename.startswith(prefix):
            rest = basename[len(prefix):]
            break
    else:
        return basename
    parts = rest.split('_')
    if len(parts) >= 2:
        return '_'.join(parts[1:])
    # Case without underscore after timestamp (e.g. '...091815Fast')
    m = re.search(r'\d+(.+)', rest)
    return m.group(1) if m else rest

In [ ]:
def load_spike_dataset(filepath):
    """Loads a spike dataset, returns a DataFrame with label + N1-N5."""
    col_names = ['timestamp', 'label', 'N1', 'N2', 'N3', 'N4', 'N5', 'sum']
    fixed = fix_decimal_commas(filepath)
    if fixed is not None:
        text = chr(10).join(fixed)  # chr(10) = newline
        df = pd.read_csv(StringIO(text), header=None, names=col_names)
    else:
        df = pd.read_csv(filepath, header=None, names=col_names)
    df = df.dropna()
    df = df.drop(columns=['timestamp', 'sum'])
    df['label'] = df['label'].astype(int)
    return df

---
## 3. Pipeline: Grid Search → Training → Saving

For each neuron:
1. `GridSearchCV` scoring on **accuracy** and **f1_macro** (5-Fold)
2. Best parameter selection (refit on accuracy)
3. Final training on the entire dataset
4. Export `.pkl`

In [ ]:
def train_and_save(data_dir, output_subdir, config_label,
                   spike_prefix='Spike', neuron_cols=None):
    """Grid Search + train + save for each neuron in the directory."""
    if neuron_cols is None:
        neuron_cols = NEURON_COLS
    files = sorted(glob.glob(os.path.join(data_dir, spike_prefix + '*.csv')))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []

    for filepath in files:
        name = extract_neuron_name(filepath)
        df = load_spike_dataset(filepath)
        X = df[neuron_cols].values
        y = df['label'].values

        # Grid Search with double scoring
        grid = GridSearchCV(
            estimator=RandomForestClassifier(
                random_state=42, n_jobs=-1, class_weight='balanced'),
            param_grid=PARAM_GRID,
            cv=skf,
            scoring={'accuracy': 'accuracy', 'f1': 'f1_macro'},
            refit='accuracy',
            n_jobs=-1,
            return_train_score=False
        )
        grid.fit(X, y)

        bp = grid.best_params_
        bi = grid.best_index_
        acc_mean = grid.cv_results_['mean_test_accuracy'][bi]
        acc_std  = grid.cv_results_['std_test_accuracy'][bi]
        f1_mean  = grid.cv_results_['mean_test_f1'][bi]
        f1_std   = grid.cv_results_['std_test_f1'][bi]

        # Final model on the whole dataset
        rf_final = RandomForestClassifier(
            **bp, random_state=42, n_jobs=-1, class_weight='balanced')
        rf_final.fit(X, y)

        # Saving
        out_dir = os.path.join(OUTPUT_DIR, output_subdir)
        os.makedirs(out_dir, exist_ok=True)
        pkl = os.path.join(out_dir, f'rf_model_{name}.pkl')
        joblib.dump(rf_final, pkl)

        depth_str = str(bp['max_depth']) if bp['max_depth'] is not None else 'None'
        print(f'  {name:<20s} | Acc: {acc_mean:.4f} \u00b1 {acc_std:.4f} | '
              f'F1: {f1_mean:.4f} | '
              f'n_est={bp["n_estimators"]:>3}, depth={depth_str:<4}, '
              f'split={bp["min_samples_split"]}, leaf={bp["min_samples_leaf"]}')

        results.append({
            'Neuron': name, 'Config': config_label,
            'Accuracy': acc_mean, 'Acc_Std': acc_std,
            'F1_Macro': f1_mean, 'F1_Std': f1_std,
            'n_estimators': bp['n_estimators'],
            'max_depth': bp['max_depth'],
            'min_samples_split': bp['min_samples_split'],
            'min_samples_leaf': bp['min_samples_leaf'],
            'Samples': len(df)
        })

    return results

---
## 4. Dataset WITH Rooting
Data generated with rooting transformation (spatial normalization relative to the wrist).

In [ ]:
print('=' * 80)
print('  GRID SEARCH + TRAINING \u2014 WITH ROOTING')
print('=' * 80)
print()
results_R = train_and_save(ROOTING_DIR, 'model_blind', 'Rooting',
                           spike_prefix='SpikeB_')
print(f'{chr(10)}\u2714 {len(results_R)} models saved')

### 4.1 Grid Search Results — With Rooting

In [ ]:
df_R = pd.DataFrame(results_R).sort_values('Accuracy', ascending=False)
df_R[['Neuron','Accuracy','Acc_Std','F1_Macro','F1_Std',
      'n_estimators','max_depth','min_samples_split','min_samples_leaf','Samples']].style.format({
    'Accuracy': '{:.4f}', 'Acc_Std': '{:.4f}',
    'F1_Macro': '{:.4f}', 'F1_Std': '{:.4f}'
}).background_gradient(subset=['Accuracy'], cmap='Greens')

---
## 5. Dataset WITHOUT Rooting
Data generated without spatial normalization.

In [ ]:
print('=' * 80)
print('  GRID SEARCH + TRAINING \u2014 WITHOUT ROOTING')
print('=' * 80)
print()
results_NR = train_and_save(NOROOTING_DIR, 'model_route_node', 'No Rooting',
                            spike_prefix='SpikeN_')
print(f'{chr(10)}\u2714 {len(results_NR)} models saved')

### 5.1 Grid Search Results — Without Rooting

In [ ]:
df_NR = pd.DataFrame(results_NR).sort_values('Accuracy', ascending=False)
df_NR[['Neuron','Accuracy','Acc_Std','F1_Macro','F1_Std',
       'n_estimators','max_depth','min_samples_split','min_samples_leaf','Samples']].style.format({
    'Accuracy': '{:.4f}', 'Acc_Std': '{:.4f}',
    'F1_Macro': '{:.4f}', 'F1_Std': '{:.4f}'
}).background_gradient(subset=['Accuracy'], cmap='Blues')

---
## 6. Comparative Plots

In [ ]:
# Preparation
order = df_R.sort_values('Accuracy', ascending=False)['Neuron'].tolist()
x = np.arange(len(order))
w = 0.35

def get_vals(df, col, neurons):
    return [df[df['Neuron'] == n][col].values[0]
            if n in df['Neuron'].values else 0 for n in neurons]

ar, sr = get_vals(df_R, 'Accuracy', order), get_vals(df_R, 'Acc_Std', order)
anr, snr = get_vals(df_NR, 'Accuracy', order), get_vals(df_NR, 'Acc_Std', order)

# --- PLOT 1: Accuracy with error bars ---
fig, ax = plt.subplots(figsize=(14, 7))

b1 = ax.bar(x - w/2, ar, w, yerr=sr, label='With Rooting',
            color='#27ae60', edgecolor='white', capsize=4, alpha=0.9)
b2 = ax.bar(x + w/2, anr, w, yerr=snr, label='Without Rooting',
            color='#2980b9', edgecolor='white', capsize=4, alpha=0.9)

for b, a in zip(b1, ar):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.012,
            f'{a:.3f}', ha='center', fontsize=8, fontweight='bold')
for b, a in zip(b2, anr):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.012,
            f'{a:.3f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(order, rotation=45, ha='right', fontsize=11)
ax.set_ylabel('Accuracy (5-Fold CV)', fontsize=12)
ax.set_title('Accuracy Comparison per Neuron Type\n'
             '(Optimized Grid Search, features: N1\u2013N5)',
             fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.12)
ax.axhline(y=1/6, color='grey', ls='--', alpha=0.5, label='Chance (1/6)')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'accuracy_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- PLOT 2: Delta Rooting ---
delta = [a - b for a, b in zip(ar, anr)]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#27ae60' if d >= 0 else '#c0392b' for d in delta]
bars = ax.bar(order, delta, color=colors, edgecolor='white', alpha=0.85)
for b, d in zip(bars, delta):
    va = 'bottom' if d >= 0 else 'top'
    off = 0.002 if d >= 0 else -0.008
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+off,
            f'{d:+.3f}', ha='center', va=va, fontsize=9, fontweight='bold')
ax.axhline(y=0, color='black', lw=0.8)
ax.set_ylabel('\u0394 Accuracy (Rooting \u2212 No Rooting)', fontsize=11)
ax.set_title('Rooting Impact on Classification', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'delta_rooting.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Summary Table and Excel Export

In [ ]:
df_all = pd.concat([df_R, df_NR], ignore_index=True)

# Pivot for comparison
cols = ['Neuron','Accuracy','Acc_Std','F1_Macro','n_estimators','max_depth','min_samples_split','min_samples_leaf']
pr = df_R[cols].rename(columns={c: c+'_R' if c != 'Neuron' else c for c in cols})
pnr = df_NR[cols].rename(columns={c: c+'_NR' if c != 'Neuron' else c for c in cols})
df_cmp = pd.merge(pr, pnr, on='Neuron', how='outer')
df_cmp['Delta_Acc'] = df_cmp['Accuracy_R'] - df_cmp['Accuracy_NR']
df_cmp = df_cmp.sort_values('Accuracy_R', ascending=False)

# Save Excel
xlsx = os.path.join(OUTPUT_DIR, 'summary_results.xlsx')
with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
    df_cmp.to_excel(w, sheet_name='Comparison', index=False)
    df_R.to_excel(w, sheet_name='Rooting', index=False)
    df_NR.to_excel(w, sheet_name='No_Rooting', index=False)
    df_all.to_excel(w, sheet_name='All', index=False)

print(f'\u2714 Excel saved: {xlsx}')
print('  Sheets: Comparison, Rooting, No_Rooting, All')
print()

df_cmp[['Neuron','Accuracy_R','Acc_Std_R','Accuracy_NR','Acc_Std_NR','Delta_Acc']].style.format({
    'Accuracy_R': '{:.4f}', 'Acc_Std_R': '{:.4f}',
    'Accuracy_NR': '{:.4f}', 'Acc_Std_NR': '{:.4f}',
    'Delta_Acc': '{:+.4f}'
}).background_gradient(subset=['Delta_Acc'], cmap='RdYlGn', vmin=0)

---
## 8. Generated `.pkl` Models
Ready for `joblib.load()` in LabVIEW Python Node.

In [ ]:
for label, subdir in [('WITH ROOTING', 'model_blind'), ('WITHOUT ROOTING', 'model_route_node')]:
    pkls = sorted(glob.glob(os.path.join(OUTPUT_DIR, subdir, '*.pkl')))
    print(f'--- {label} ({len(pkls)} models) ---')
    for f in pkls:
        print(f'  {os.path.basename(f):40s} ({os.path.getsize(f)/1024:>6.0f} KB)')
    print()